In [2]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

In [4]:
load_dotenv(override=True)
api_key=os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

API key looks good so far


In [5]:
MODEL='gpt-5-nano'
openai=OpenAI()

In [9]:
from bs4 import BeautifulSoup
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


In [11]:
links=fetch_website_links("https://myknowtech.com")
links

['https://myknowtech.com',
 'https://myknowtech.com/product/secops-studio/',
 'https://myknowtech.com/solution/stack-shifter/',
 'https://myknowtech.com/services',
 'https://myknowtech.com/service/pega-enablement/',
 'https://myknowtech.com/service/pega-modernization-service/',
 'https://myknowtech.com/about/',
 'https://myknowtech.com/contact/',
 'https://myknowtech.com/blog',
 '#ekit_modal-popup-7c4ec3bb',
 'https://myknowtech.com/contact/',
 '#',
 '#',
 '#',
 '#',
 '#',
 '#',
 'https://myknowtech.com/solution/stack-shifter/',
 'https://myknowtech.com/service/pega-modernization-service',
 'https://myknowtech.com/service/pega-enablement',
 'https://myknowtech.com/blog/code-vault/add-a-pega-icon-font-in-traditional-ui/',
 'https://myknowtech.com/blog/code-vault/add-a-pega-icon-font-in-traditional-ui/',
 'https://myknowtech.com/blog/author/premkumar/',
 'https://myknowtech.com/topics/code-vault/',
 'https://myknowtech.com/blog/code-vault/pega-using-azure-key-vault-as-external-secret-sto

In [12]:
link_system_prompt="""
You are provided with list of links found on a web page.You are able to decide which links are relavent to Website to include in the Brochure about the 
company.Those links are such as About Page, or Company Page, or Careers/Job Page.
You should respond in JSON as below.

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [13]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [14]:
print(get_links_user_prompt('https://myknowtech.com'))


Here is the list of links on the website https://myknowtech.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://myknowtech.com
https://myknowtech.com/product/secops-studio/
https://myknowtech.com/solution/stack-shifter/
https://myknowtech.com/services
https://myknowtech.com/service/pega-enablement/
https://myknowtech.com/service/pega-modernization-service/
https://myknowtech.com/about/
https://myknowtech.com/contact/
https://myknowtech.com/blog
#ekit_modal-popup-7c4ec3bb
https://myknowtech.com/contact/
#
#
#
#
#
#
https://myknowtech.com/solution/stack-shifter/
https://myknowtech.com/service/pega-modernization-service
https://myknowtech.com/service/pega-enablement
https://myknowtech.com/blog/code-vault/add-a-pega-icon-font-in-traditional-ui/
https://myknowtech.com/blog/code-vault/add-a-pega-icon-f

In [27]:
def select_relavent_links(url):
    response=openai.chat.completions.create(model=MODEL,
                                            messages=[{"role":"system","content":link_system_prompt},
                                                      {"role":"user","content":get_links_user_prompt(url)}],
                                            response_format={"type":"json_object"}
                                           )
    result=response.choices[0].message.content
    links=json.loads(result)
    return links
                                          
    

In [18]:
select_relavent_links('https://myknowtech.com')

{'links': [{'type': 'about page', 'url': 'https://myknowtech.com/about/'},
  {'type': 'leadership page', 'url': 'https://myknowtech.com/leadership'},
  {'type': 'contact page', 'url': 'https://myknowtech.com/contact/'},
  {'type': 'homepage', 'url': 'https://myknowtech.com'}]}

In [24]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [30]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relavent_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [31]:
print(fetch_page_and_all_relevant_links("https://myknowtech.com"))

## Landing Page:

MyKnowTech - A Premium Pega Consulting Company

Products
SecOps Studio
Solutions
Pega Stack Shifter
Services
Pega Enablement
Pega Modernization
About
Contact
Blog
X
Are you
planning Constellation migration
struggling to establize Pega COE
worried about security risks
in your company?
We help you navigate challenges with tailor-made Pega solutions, maximizing the value of your investment in Pega.
Your success is our business.
Lets discuss
WHO WE ARE?
Passionate Pega experts dedicated to creating meaningful impacts with empathy and expertise.
WHAT WE DO?
Deliver specialized Pega solutions and tailored services to meet unique business needs.
HOW WE DO?
Take ownership, collaborate closely, and ensure impactful, sustainable results throughout.
We help you to
maximize
the value of your investment in Pega.
Navigate your challenges with our tailor-made Pega solutions and Services.
WHO WE ARE?
Passionate Pega experts dedicated to creating meaningful impacts with empathy and ex

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [32]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [33]:
get_brochure_user_prompt("MyKnowTech", "https://myknowtech.com")

'\nYou are looking at a company called: MyKnowTech\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nMyKnowTech - A Premium Pega Consulting Company\n\nProducts\nSecOps Studio\nSolutions\nPega Stack Shifter\nServices\nPega Enablement\nPega Modernization\nAbout\nContact\nBlog\nX\nAre you\nplanning Constellation migration\nstruggling to establize Pega COE\nworried about security risks\nin your company?\nWe help you navigate challenges with tailor-made Pega solutions, maximizing the value of your investment in Pega.\nYour success is our business.\nLets discuss\nWHO WE ARE?\nPassionate Pega experts dedicated to creating meaningful impacts with empathy and expertise.\nWHAT WE DO?\nDeliver specialized Pega solutions and tailored services to meet unique business needs.\nHOW WE DO?\nTake ownership, collaborate closely, and ensure impactful, sustainable resu

In [34]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [36]:
create_brochure("MyKnowTech", "https://myknowtech.com")

NameError: name 'Markdown' is not defined

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("MyKnowTech", "https://myknowtech.com")